In [20]:
# plot_roll_csv.py
# -----------------------------------------------------------------------------
# Plot and analyze gyro-only roll logs from your ESP32 (LSM9DS1) CSV dump.
#
# What this script does:
#  1) Loads roll.csv (even if it contains miniterm banners / --- CSV DUMP ... --- lines)
#  2) Estimates the gyroscope roll-rate bias (offset) from an initial "still" window
#  3) Plots:
#       - roll_gyro_deg vs time
#       - omega_roll_deg_s and gx/gy/gz vs time
#       - Histogram of omega_roll_deg_s in the bias window
#       - (Important) A *proper* "debiased roll" computed using the SAME dt samples
#
# Why "proper debiased roll" matters:
#  Your firmware integrates only when a gyro sample is actually read:
#      theta[k+1] = theta[k] + omega[k] * dt[k]
#  So to remove bias consistently, do:
#      theta_debiased[k] = Σ_{i=1..k} (omega[i] - b_hat) * dt[i]
#
#  where b_hat is the estimated constant bias (deg/s) from an initial stationary window.
#
# Usage:
#   python3 plot_roll_csv.py /path/to/roll.csv
#
# Dependencies:
#   pip install pandas numpy plotly
# -----------------------------------------------------------------------------

from __future__ import annotations

import sys
from io import StringIO

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px


# -----------------------------------------------------------------------------
# User settings (adjust as needed)
# -----------------------------------------------------------------------------
BIAS_WINDOW_S = 20.0   # seconds used to estimate bias (keep IMU still for this duration)
SKIP_FIRST_S = 0.5     # seconds to ignore at start (startup transient)
TEMPLATE = "plotly_white"

# A clean, readable palette (Plotly will cycle through these in order)
COLORWAY = ["#2F80ED", "#27AE60", "#EB5757", "#9B51E0", "#F2994A", "#56CCF2"]

# Optional: remove obvious outliers in bias window (helps if you bumped the sensor)
# Set to None to disable. Typical values: 3.0 to 5.0
BIAS_CLIP_STD = 4.0


# -----------------------------------------------------------------------------
# CSV loading helper
# -----------------------------------------------------------------------------
def load_csv_clean(path: str) -> pd.DataFrame:
    """
    Loads the CSV while ignoring miniterm banners / dump markers / accidental plotter lines.

    Your serial dump can contain lines like:
      --- Miniterm on /dev/ttyUSB0 ...
      --- CSV DUMP START ---
      --- CSV DUMP END ---
    and sometimes "roll_gyro_deg:xx" if you accidentally captured plotter output.

    We strip those and parse only actual CSV rows.
    """
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    cleaned_lines: list[str] = []
    for ln in lines:
        s = ln.strip()
        if not s:
            continue
        # Skip any banner/marker lines
        if s.startswith("---"):
            continue
        # Skip plotter-only format lines (if accidentally captured)
        if s.startswith("roll_gyro_deg:"):
            continue
        cleaned_lines.append(ln)

    df = pd.read_csv(StringIO("".join(cleaned_lines)))

    expected = [
        "time_s", "dt_s",
        "gx_deg_s", "gy_deg_s", "gz_deg_s",
        "omega_roll_deg_s",
        "roll_gyro_deg",
    ]
    missing = [c for c in expected if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in CSV: {missing}\nFound: {list(df.columns)}")

    # Sort and reset index for safety
    df = df.sort_values("time_s").reset_index(drop=True)

    # Ensure numeric types
    for c in expected:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Drop rows with NaNs in critical columns
    df = df.dropna(subset=["time_s", "dt_s", "omega_roll_deg_s", "roll_gyro_deg"]).reset_index(drop=True)

    return df


# -----------------------------------------------------------------------------
# Bias (offset) estimation
# -----------------------------------------------------------------------------
def estimate_bias(df: pd.DataFrame) -> dict:
    """
    Estimate gyro roll-rate bias b_hat from an initial window where the IMU is still.

    We assume the measurement model (for roll rate):
        omega_meas(t) = omega_true(t) + b + noise

    When the IMU is still (omega_true ≈ 0), then:
        omega_meas ≈ b + noise
    So a good bias estimate is the mean of omega_meas over a stationary window.

    Returns stats including:
        bias_mean_deg_s, bias_median_deg_s, bias_std_deg_s,
        drift_deg_per_min, predicted_drift_60s
    """
    t0 = float(df["time_s"].min())
    mask = (df["time_s"] >= t0 + SKIP_FIRST_S) & (df["time_s"] <= t0 + BIAS_WINDOW_S)

    w = df.loc[mask, "omega_roll_deg_s"].to_numpy()
    if w.size < 10:
        raise ValueError(
            "Not enough samples in the bias window.\n"
            "Increase BIAS_WINDOW_S or check that your CSV contains data early on."
        )

    # Optional robust clipping (remove occasional spikes if you moved the board)
    if BIAS_CLIP_STD is not None and w.size >= 20:
        mu = np.mean(w)
        sigma = np.std(w, ddof=1) if w.size > 1 else 0.0
        if sigma > 1e-12:
            keep = np.abs(w - mu) <= (BIAS_CLIP_STD * sigma)
            w = w[keep]

    bias_mean = float(np.mean(w))
    bias_median = float(np.median(w))
    bias_std = float(np.std(w, ddof=1)) if w.size > 1 else 0.0

    drift_deg_per_min = bias_mean * 60.0
    predicted_drift_60s = bias_mean * 60.0

    return {
        "bias_mean_deg_s": bias_mean,
        "bias_median_deg_s": bias_median,
        "bias_std_deg_s": bias_std,
        "drift_deg_per_min": drift_deg_per_min,
        "predicted_drift_60s_deg": predicted_drift_60s,
        "n_bias_samples": int(w.size),
        "bias_window_used_s": float(BIAS_WINDOW_S),
        "skip_first_s": float(SKIP_FIRST_S),
    }


# -----------------------------------------------------------------------------
# Proper debiased integration using recorded dt_s
# -----------------------------------------------------------------------------
def compute_debiased_roll(df: pd.DataFrame, bias_deg_s: float) -> np.ndarray:
    """
    Compute a *proper* debiased roll estimate using the *same dt_s samples* you integrated.

    Your firmware effectively does:
        theta[k] = theta[k-1] + omega[k] * dt[k]

    If omega[k] = omega_true[k] + b + noise, then integrating omega causes drift b*t.
    To remove bias consistently with your discrete integration, do:

        theta_debiased[k] = Σ_{i=1..k} (omega[i] - b_hat) * dt[i]

    This uses dt[i] from the CSV, so it remains consistent even if the IMU sampling
    is not perfectly uniform.
    """
    omega = df["omega_roll_deg_s"].to_numpy()
    dt = df["dt_s"].to_numpy()

    # elementwise: (omega - bias) * dt  -> deg
    increments = (omega - bias_deg_s) * dt
    theta_debiased = np.cumsum(increments)

    return theta_debiased


# -----------------------------------------------------------------------------
# Plotting helpers
# -----------------------------------------------------------------------------
def plot_roll_timeseries(df: pd.DataFrame, theta_debiased: np.ndarray):
    fig = go.Figure()
    fig.update_layout(template=TEMPLATE, colorway=COLORWAY)

    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_gyro_deg"],
        mode="lines", name="roll_gyro_deg (firmware integration)",
        line=dict(width=2)
    ))

    fig.add_trace(go.Scatter(
        x=df["time_s"], y=theta_debiased,
        mode="lines", name="roll_debiased (Σ (ω - b̂) dt)",
        line=dict(width=2, dash="dash")
    ))

    fig.update_layout(
        title="Gyro-only Roll vs Time (Integrated) + Proper Debiased Roll",
        xaxis_title="time (s)",
        yaxis_title="roll (deg)",
        hovermode="x unified",
        legend_title="signals",
    )
    fig.show()


def plot_gyro_rates(df: pd.DataFrame):
    fig = go.Figure()
    fig.update_layout(template=TEMPLATE, colorway=COLORWAY)

    for col in ["gx_deg_s", "gy_deg_s", "gz_deg_s", "omega_roll_deg_s"]:
        fig.add_trace(go.Scatter(
            x=df["time_s"], y=df[col],
            mode="lines", name=col,
            line=dict(width=2)
        ))

    fig.update_layout(
        title="Gyroscope Angular Rates vs Time",
        xaxis_title="time (s)",
        yaxis_title="angular rate (deg/s)",
        hovermode="x unified",
        legend_title="channels",
    )
    fig.show()


def plot_bias_window_hist(df: pd.DataFrame):
    t0 = float(df["time_s"].min())
    mask = (df["time_s"] >= t0 + SKIP_FIRST_S) & (df["time_s"] <= t0 + BIAS_WINDOW_S)
    w = df.loc[mask, "omega_roll_deg_s"]

    fig = px.histogram(
        w,
        nbins=50,
        template=TEMPLATE,
        title=f"Bias Window Histogram: omega_roll_deg_s (first {BIAS_WINDOW_S}s, skip {SKIP_FIRST_S}s)"
    )
    fig.update_layout(
        xaxis_title="omega_roll_deg_s (deg/s)",
        yaxis_title="count",
    )
    fig.show()


def plot_dt(df: pd.DataFrame):
    fig = go.Figure()
    fig.update_layout(template=TEMPLATE, colorway=COLORWAY)

    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["dt_s"],
        mode="lines", name="dt_s",
        line=dict(width=2)
    ))

    fig.update_layout(
        title="Recorded dt_s vs Time (Sampling / Integration Step Size)",
        xaxis_title="time (s)",
        yaxis_title="dt (s)",
        hovermode="x unified",
    )
    fig.show()


# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------
def main():
    if len(sys.argv) < 2:
        print("Usage: python3 plot_roll_csv.py /path/to/roll.csv")
        sys.exit(1)

    path = "/home/iitgn-robotics/Debojit_WS/ME692_Data_and_Observer_Design/data/rollbiased0.csv"
    df = load_csv_clean(path)

    # Estimate bias from early stationary window
    stats = estimate_bias(df)
    bias_mean = stats["bias_mean_deg_s"]

    # Proper debiased roll using dt samples
    theta_debiased = compute_debiased_roll(df, bias_deg_s=bias_mean)

    # Print summary
    print("\n=== Gyro Roll-Rate Offset (Bias) Estimate ===")
    print(f"Bias window: first {stats['bias_window_used_s']}s (skipping first {stats['skip_first_s']}s)")
    print(f"Samples used: {stats['n_bias_samples']}")
    print(f"Mean bias      : {stats['bias_mean_deg_s']:+.6f} deg/s")
    print(f"Median bias    : {stats['bias_median_deg_s']:+.6f} deg/s")
    print(f"Std dev (noise): {stats['bias_std_deg_s']:.6f} deg/s")
    print(f"Drift rate (if uncorrected): {stats['drift_deg_per_min']:+.3f} deg/min")
    print(f"Predicted drift over 60s    : {stats['predicted_drift_60s_deg']:+.3f} deg")

    # Extra: compare final values (helps sanity-check)
    print("\n=== End-of-run sanity check ===")
    print(f"Final firmware roll_gyro_deg       : {float(df['roll_gyro_deg'].iloc[-1]):+.3f} deg")
    print(f"Final proper debiased roll (script): {float(theta_debiased[-1]):+.3f} deg")

    # Plots
    plot_roll_timeseries(df, theta_debiased)
    plot_gyro_rates(df)
    plot_bias_window_hist(df)
    plot_dt(df)


if __name__ == "__main__":
    main()



=== Gyro Roll-Rate Offset (Bias) Estimate ===
Bias window: first 20.0s (skipping first 0.5s)
Samples used: 1843
Mean bias      : +1.808436 deg/s
Median bias    : +1.831055 deg/s
Std dev (noise): 0.067425 deg/s
Drift rate (if uncorrected): +108.506 deg/min
Predicted drift over 60s    : +108.506 deg

=== End-of-run sanity check ===
Final firmware roll_gyro_deg       : +49.863 deg
Final proper debiased roll (script): -0.076 deg


In [19]:
# plot_roll_unbiased_csv.py
# -----------------------------------------------------------------------------
# Plot and analyze the "biased + unbiased" roll log produced by your ESP32 code:
#
# CSV columns (from your firmware):
#   time_s, dt_s, omega_meas_deg_s, omega_corr_deg_s, roll_biased_deg, roll_unbiased_deg
#
# Assumption for this script (as you asked):
#   - The IMU was kept STILL for the ENTIRE duration.
#   - Therefore, omega_true ≈ 0 for all time, so omega_meas ≈ bias + noise.
#   - We estimate a single constant bias from the full dataset:
#         b_hat = mean(omega_meas_deg_s)
#
# We then:
#   1) Plot raw omega_meas and omega_corr vs time
#   2) Plot roll_biased and roll_unbiased (firmware) vs time
#   3) Recompute our own unbiased roll using the same dt samples:
#         theta_unbiased_recomputed[k] = Σ (omega_meas[k] - b_hat) * dt[k]
#      (This is the mathematically clean discrete-time bias correction)
#   4) Show histogram of omega_meas (noise distribution around bias)
#
# Usage:
#   python3 plot_roll_unbiased_csv.py /path/to/roll_unbiased0.csv
#
# Dependencies:
#   python3 -m pip install --user numpy pandas plotly
# -----------------------------------------------------------------------------

from __future__ import annotations

import sys
from io import StringIO

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px


# ----------------------------- Styling -----------------------------
TEMPLATE = "plotly_white"
COLORWAY = ["#2F80ED", "#27AE60", "#EB5757", "#9B51E0", "#F2994A", "#56CCF2"]
# ------------------------------------------------------------------


def load_csv_clean(path: str) -> pd.DataFrame:
    """
    Loads the CSV while removing miniterm banners / dump markers, if present.
    """
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    cleaned = []
    for ln in lines:
        s = ln.strip()
        if not s:
            continue
        if s.startswith("---"):  # miniterm banner / CSV DUMP markers
            continue
        cleaned.append(ln)

    df = pd.read_csv(StringIO("".join(cleaned)))

    expected = [
        "time_s",
        "dt_s",
        "omega_meas_deg_s",
        "omega_corr_deg_s",
        "roll_biased_deg",
        "roll_unbiased_deg",
    ]
    missing = [c for c in expected if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in CSV: {missing}\nFound: {list(df.columns)}")

    # Convert to numeric and drop bad rows
    for c in expected:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=expected).sort_values("time_s").reset_index(drop=True)

    return df


def estimate_bias_full_run(df: pd.DataFrame) -> dict:
    """
    Under the "still for all time" assumption:
      omega_true ≈ 0  -> omega_meas ≈ b + noise
    So b_hat = mean(omega_meas).

    Also compute a noise std estimate around that mean.
    """
    w = df["omega_meas_deg_s"].to_numpy()
    b_hat = float(np.mean(w))
    med = float(np.median(w))
    std = float(np.std(w, ddof=1)) if w.size > 1 else 0.0

    # Drift rate if you integrate the bias (deg/min)
    drift_deg_per_min = b_hat * 60.0

    # Predicted drift over the whole recorded interval:
    # Use total elapsed time from the data itself
    T = float(df["time_s"].iloc[-1] - df["time_s"].iloc[0])
    predicted_drift_deg = b_hat * T

    return {
        "bias_mean_deg_s": b_hat,
        "bias_median_deg_s": med,
        "noise_std_deg_s": std,
        "drift_deg_per_min": drift_deg_per_min,
        "T_total_s": T,
        "predicted_drift_deg": predicted_drift_deg,
        "n_samples": int(w.size),
    }


def recompute_unbiased_roll(df: pd.DataFrame, bias_deg_s: float) -> np.ndarray:
    """
    Recompute an unbiased roll angle using discrete integration with recorded dt:

      theta_unbiased[k] = Σ_{i=1..k} (omega_meas[i] - b_hat) * dt[i]

    This is the clean discrete-time bias correction consistent with your integration model.
    """
    omega = df["omega_meas_deg_s"].to_numpy()
    dt = df["dt_s"].to_numpy()
    return np.cumsum((omega - bias_deg_s) * dt)


def plot_rates(df: pd.DataFrame, bias_deg_s: float):
    """
    Plot omega_meas and omega_corr. Also draw a horizontal line at the estimated bias.
    """
    fig = go.Figure()
    fig.update_layout(template=TEMPLATE, colorway=COLORWAY)

    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["omega_meas_deg_s"],
        mode="lines", name="omega_meas_deg_s (raw gyro)",
        line=dict(width=2)
    ))

    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["omega_corr_deg_s"],
        mode="lines", name="omega_corr_deg_s (firmware: omega - bias_const)",
        line=dict(width=2)
    ))

    # Estimated bias line
    fig.add_trace(go.Scatter(
        x=[df["time_s"].iloc[0], df["time_s"].iloc[-1]],
        y=[bias_deg_s, bias_deg_s],
        mode="lines", name=f"b_hat = mean(omega_meas) = {bias_deg_s:.4f} deg/s",
        line=dict(width=2, dash="dash")
    ))

    fig.update_layout(
        title="Gyro Angular Rate vs Time (Still Run)",
        xaxis_title="time (s)",
        yaxis_title="angular rate (deg/s)",
        hovermode="x unified",
        legend_title="signals",
    )
    fig.show()


def plot_rolls(df: pd.DataFrame, roll_unbiased_recomputed: np.ndarray):
    """
    Plot firmware integrated roll (biased & unbiased) and compare with recomputed unbiased roll.
    """
    fig = go.Figure()
    fig.update_layout(template=TEMPLATE, colorway=COLORWAY)

    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_biased_deg"],
        mode="lines", name="roll_biased_deg (firmware)",
        line=dict(width=2)
    ))

    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_unbiased_deg"],
        mode="lines", name="roll_unbiased_deg (firmware)",
        line=dict(width=2)
    ))

    fig.add_trace(go.Scatter(
        x=df["time_s"], y=roll_unbiased_recomputed,
        mode="lines", name="roll_unbiased_recomputed (Σ (ω - b_hat) dt)",
        line=dict(width=2, dash="dash")
    ))

    fig.update_layout(
        title="Integrated Roll vs Time (Still Run)",
        xaxis_title="time (s)",
        yaxis_title="roll (deg)",
        hovermode="x unified",
        legend_title="signals",
    )
    fig.show()


def plot_hist(df: pd.DataFrame, bias_deg_s: float):
    """
    Histogram of omega_meas with a vertical line at bias mean.
    """
    fig = px.histogram(
        df, x="omega_meas_deg_s", nbins=60,
        template=TEMPLATE,
        title="Histogram of omega_meas_deg_s (Still Run)"
    )
    fig.update_layout(xaxis_title="omega_meas_deg_s (deg/s)", yaxis_title="count")

    fig.add_vline(
        x=bias_deg_s,
        line_width=2,
        line_dash="dash",
        annotation_text=f"mean bias = {bias_deg_s:.4f} deg/s",
        annotation_position="top right",
    )
    fig.show()


def plot_dt(df: pd.DataFrame):
    """
    dt sanity plot. For a stable IMU update, dt should hover around a consistent value.
    """
    fig = go.Figure()
    fig.update_layout(template=TEMPLATE, colorway=COLORWAY)
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["dt_s"],
        mode="lines", name="dt_s",
        line=dict(width=2)
    ))
    fig.update_layout(
        title="dt_s vs Time (Gyro Sample Interval)",
        xaxis_title="time (s)",
        yaxis_title="dt (s)",
        hovermode="x unified",
    )
    fig.show()


def main():
    if len(sys.argv) < 2:
        print("Usage: python3 plot_roll_unbiased_csv.py /path/to/roll_unbiased0.csv")
        sys.exit(1)

    path = "/home/iitgn-robotics/Debojit_WS/ME692_Data_and_Observer_Design/data/roll_unbiased0.csv"
    df = load_csv_clean(path)

    stats = estimate_bias_full_run(df)
    b_hat = stats["bias_mean_deg_s"]

    roll_unbiased_recomputed = recompute_unbiased_roll(df, bias_deg_s=b_hat)

    # ----------------------------- Print summary -----------------------------
    print("\n=== Bias Estimate (Assuming STILL for Entire Run) ===")
    print(f"Samples used            : {stats['n_samples']}")
    print(f"Total duration (s)      : {stats['T_total_s']:.3f}")
    print(f"Mean bias b_hat         : {stats['bias_mean_deg_s']:+.6f} deg/s")
    print(f"Median                  : {stats['bias_median_deg_s']:+.6f} deg/s")
    print(f"Noise std (around mean) : {stats['noise_std_deg_s']:.6f} deg/s")
    print(f"Drift rate if uncorrected: {stats['drift_deg_per_min']:+.3f} deg/min")
    print(f"Predicted drift over run : {stats['predicted_drift_deg']:+.3f} deg")

    print("\n=== End-of-run sanity check ===")
    print(f"Final roll_biased_deg (firmware)        : {float(df['roll_biased_deg'].iloc[-1]):+.3f} deg")
    print(f"Final roll_unbiased_deg (firmware)      : {float(df['roll_unbiased_deg'].iloc[-1]):+.3f} deg")
    print(f"Final roll_unbiased_recomputed (script) : {float(roll_unbiased_recomputed[-1]):+.3f} deg")

    # ----------------------------- Plots -----------------------------
    plot_rates(df, bias_deg_s=b_hat)
    plot_rolls(df, roll_unbiased_recomputed)
    plot_hist(df, bias_deg_s=b_hat)
    plot_dt(df)


if __name__ == "__main__":
    main()



=== Bias Estimate (Assuming STILL for Entire Run) ===
Samples used            : 300
Total duration (s)      : 11.570
Mean bias b_hat         : +1.783447 deg/s
Median                  : +1.770020 deg/s
Noise std (around mean) : 0.068579 deg/s
Drift rate if uncorrected: +107.007 deg/min
Predicted drift over run : +20.634 deg

=== End-of-run sanity check ===
Final roll_biased_deg (firmware)        : +16.071 deg
Final roll_unbiased_deg (firmware)      : -0.292 deg
Final roll_unbiased_recomputed (script) : -0.013 deg
